In [1]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

load_dotenv(override=True)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
    persist_directory="../vector_db",
    embedding_function=embeddings,
)

print("Chunks in database:", vectorstore._collection.count())

Chunks in database: 14


In [2]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
llm = ChatOpenAI(model="gpt-4.1-nano")

SYSTEM_PROMPT = """You are an assistant for BrightDesk IT Solutions.
Answer questions using ONLY the context provided.
If the context contains several people or services that match the question,
briefly describe each one.
If the answer is not in the context at all, say you don't know."""


def answer_question(question):
    docs = retriever.invoke(question)

    print("Retrieved:")
    for doc in docs:
        print("  -", doc.metadata["section"])

    context = ""
    for doc in docs:
        context = context + doc.page_content + "\n\n"

    messages = [
        ("system", SYSTEM_PROMPT),
        ("user", "Context:\n" + context + "\nQuestion: " + question),
    ]
    return llm.invoke(messages).content 



In [3]:
print(answer_question("What does Priya do?"))
print("=========")
print(answer_question("When did she join?"))

Retrieved:
  - Employee: Priya Patel
  - Employee: Sophie Wright
  - Employee: Rahul Patel
Priya Patel is the Head of Service Desk. She manages the 12-person service desk team, owns the ticketing process, and reports monthly on response times.
Retrieved:
  - Employee: Sophie Wright
  - Employee: Emma Clarke
  - Employee: Priya Patel
You didn't specify which person you are referring to. Could you please clarify?


In [9]:
def answer_question(question, history):
    docs = retriever.invoke(question)

    print("Retrieved:")
    for doc in docs:
        print("  -", doc.metadata["section"])

    context = ""
    for doc in docs:
        context = context + doc.page_content + "\n\n"

    messages = [("system", SYSTEM_PROMPT)]
    for role, text in history:
        messages.append((role, text))
    messages.append(("user", "Context:\n" + context + "\nQuestion: " + question))

    answer = llm.invoke(messages).content

    history.append(("user", question))
    history.append(("assistant", answer))
    return answer


history = []
print(answer_question("What does Priya do?", history))
print("=========")
print(answer_question("When did she join?", history))

Retrieved:
  - Employee: Priya Patel
  - Employee: Sophie Wright
  - Employee: Rahul Patel
Priya Patel is the Head of Service Desk. She manages the 12-person service desk team, owns the ticketing process, and reports monthly on response times.
Retrieved:
  - Employee: Sophie Wright
  - Employee: Emma Clarke
  - Employee: Priya Patel
Priya Patel joined in March 2019.


In [10]:
REWRITE_PROMPT = """Rewrite the user's latest question as a standalone question
that makes sense without the conversation history.
Replace pronouns like 'she', 'he', 'it' or 'that' with what they refer to.
If the question is already standalone, return it unchanged.
Return ONLY the rewritten question."""


def rewrite_question(question, history):
    if len(history) == 0:
        return question

    messages = [("system", REWRITE_PROMPT)]
    for role, text in history:
        messages.append((role, text))
    messages.append(("user", question))

    return llm.invoke(messages).content


def answer_question(question, history):
    search_question = rewrite_question(question, history)
    print("Searching for:", search_question)
    docs = retriever.invoke(search_question)

    print("Retrieved:")
    for doc in docs:
        print("  -", doc.metadata["section"])

    context = ""
    for doc in docs:
        context = context + doc.page_content + "\n\n"

    messages = [("system", SYSTEM_PROMPT)]
    for role, text in history:
        messages.append((role, text))
    messages.append(("user", "Context:\n" + context + "\nQuestion: " + question))

    answer = llm.invoke(messages).content

    history.append(("user", question))
    history.append(("assistant", answer))
    return answer


history = []
print(answer_question("What does Priya do?", history))
print("=========")
print(answer_question("When did she join?", history))

Searching for: What does Priya do?
Retrieved:
  - Employee: Priya Patel
  - Employee: Sophie Wright
  - Employee: Rahul Patel
Priya Patel, as the Head of Service Desk, manages the 12-person service desk team, owns the ticketing process, and reports monthly on response times.
Searching for: When did Priya Patel join the company?
Retrieved:
  - Employee: Priya Patel
  - Employee: Rahul Patel
  - Employee: Sophie Wright
Priya Patel joined in March 2019.
